In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
df = pd.read_csv("Diseases_and_Symptoms_dataset.csv")

In [ ]:
print("Rows    :", df.shape[0])
print("Columns :", df.shape[1])

In [ ]:
display(df.head())

In [ ]:
print(df.columns.tolist())

In [ ]:
df.columns = (
    df.columns
    .astype(str)
    .str.strip()
)

print(df.columns.tolist())

In [ ]:
possible_disease_columns = [
    col for col in df.columns
    if col.lower() in ["diseases", "prognosis", "diagnosis", "condition"]
]

print("Possible disease columns:", possible_disease_columns)

In [ ]:
target_column = possible_disease_columns[0]

print("Target column:", target_column)

In [ ]:
X = df.drop(columns=[target_column])
y = df[target_column]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
print("Number of unique diseases:", y.nunique())

print("\nFirst 20 diseases:")
print(y.unique()[:20])

**Check Missing Values**

In [ ]:
missing = df.isnull().sum()
print("Columns with missing values:")
display(missing[missing > 0])

**Check Duplicate**

In [ ]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

In [ ]:
unique_values = set()

for column in X.columns:
    values = X[column].dropna().unique()
    unique_values.update(values)

print("Unique values found:")
print(unique_values)

In [ ]:
X = X.apply(pd.to_numeric, errors="coerce")

In [ ]:
X = X.fillna(0)

In [ ]:
print("Remaining missing values:", X.isnull().sum().sum())

**Encode disease labels**

In [ ]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Original disease:")
print(y.iloc[0])

print("\nEncoded value:")
print(y_encoded[0])

In [ ]:
for i, disease in enumerate(label_encoder.classes_[:20]):
    print(i, "→", disease)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training data :", X_train.shape)
print("Testing data  :", X_test.shape)

**Pipeline**

In [ ]:
model_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value=-1.0,
            keep_empty_features=True
        )
    ),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    )
])

print(model_pipeline)

## Fine-tune Random Forest hyperparameters

This randomized search compares a small set of tree counts, depth limits, split sizes, leaf sizes, feature sampling options, and class-weight settings. It selects the best configuration using 3-fold cross-validation and macro-F1. Run this cell before the data augmentation and final training cells.

In [ ]:
# Hyperparameter tuning on the ORIGINAL training split.
# This may take several minutes. It optimizes macro-F1 across disease classes.
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

X_train_tuning = X_train.astype(np.float32)

search_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value=-1.0,
        keep_empty_features=True
    )),
    ("classifier", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

param_distributions = {
    "classifier__n_estimators": [100, 150, 200],
    "classifier__max_depth": [None, 20, 30, 40],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", 0.5],
    "classifier__class_weight": [None, "balanced_subsample"]
}

random_search = RandomizedSearchCV(
    estimator=search_pipeline,
    param_distributions=param_distributions,
    n_iter=6,
    scoring="f1_macro",
    cv=3,
    random_state=42,
    n_jobs=1,
    verbose=2,
    refit=True,
    return_train_score=True
)

random_search.fit(X_train_tuning, y_train)

print("Best CV macro-F1:", round(random_search.best_score_, 4))
print("Best parameters:")
for name, value in random_search.best_params_.items():
    print(f"{name}: {value}")

# Use the selected hyperparameters for the final model.
best_rf_params = {
    key.replace("classifier__", ""): value
    for key, value in random_search.best_params_.items()
    if key.startswith("classifier__")
}

model_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value=-1.0,
        keep_empty_features=True
    )),
    ("classifier", RandomForestClassifier(
        **best_rf_params,
        random_state=42,
        n_jobs=-1
    ))
])

print("\nFinal model pipeline configured with tuned parameters.")

In [ ]:
# Create a partially observed version of training data
rng = np.random.default_rng(42)

X_train_array = X_train.to_numpy(dtype=np.float32).copy()

# Randomly mark 30% of symptoms as UNKNOWN (-1)
mask = rng.random(X_train_array.shape) < 0.30

X_train_masked_array = X_train_array.copy()
X_train_masked_array[mask] = -1.0

X_train_masked = pd.DataFrame(
    X_train_masked_array,
    columns=X_train.columns,
    index=X_train.index
)

# Combine original and partially observed data
X_train_augmented = pd.concat(
    [
        X_train.astype(np.float32),
        X_train_masked.astype(np.float32)
    ],
    ignore_index=True
)

y_train_augmented = np.concatenate(
    [y_train, y_train]
)

print("Original training data:", X_train.shape)
print("Augmented training data:", X_train_augmented.shape)
print("Data type:", X_train_augmented.dtypes.unique())

In [ ]:
model_pipeline.fit(
    X_train_augmented,
    y_train_augmented
)

print("Model training completed successfully!")

In [ ]:
# Convert test features to the same dtype used for training
X_test = X_test.astype(np.float32)

# Predict disease classes
y_pred = model_pipeline.predict(X_test)

print("Prediction completed!")
print("Total predictions:", len(y_pred))

**Accuracy**

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")

**Precision,Recall,F1**

In [ ]:
precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

In [ ]:
macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print(f"Macro F1: {macro_f1:.4f}")

**Probability Predictions**

In [ ]:
y_proba = model_pipeline.predict_proba(X_test)

print("Probability matrix shape:", y_proba.shape)

In [ ]:
print("Number of model classes:", len(model_pipeline.classes_))
print("Number of diseases:", len(label_encoder.classes_))

**RF Feature importance**

In [ ]:
rf = model_pipeline.named_steps["classifier"]

feature_importance = rf.feature_importances_

feature_importance_df = pd.DataFrame({
    "Symptom": X.columns,
    "Importance": feature_importance
})

feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

print(feature_importance_df.head(20).to_string(index=False))

In [ ]:
TOP_K_SYMPTOMS = 50

candidate_symptoms = (
    feature_importance_df
    .head(TOP_K_SYMPTOMS)["Symptom"]
    .tolist()
)

print("Number of candidate symptoms:", len(candidate_symptoms))
print("\nTop candidate symptoms:")

for symptom in candidate_symptoms:
    print("-", symptom)

**Function for TOP-3 Possible Conditions**

In [ ]:
def get_top_predictions(input_data, top_n=3):

    probabilities = model_pipeline.predict_proba(input_data)[0]

    top_indices = np.argsort(probabilities)[-top_n:][::-1]

    results = []

    for index in top_indices:

        disease = label_encoder.inverse_transform([index])[0]
        probability = probabilities[index] * 100

        results.append({
            "Disease": disease,
            "Probability (%)": round(probability, 2)
        })

    return pd.DataFrame(results)

**Disease-symptoms probability table**

In [ ]:
train_probability_data = X_train.copy()

train_probability_data["disease_id"] = y_train

disease_symptom_probability = (
    train_probability_data
    .groupby("disease_id")
    .mean()
)

print(
    "Disease-symptom probability table shape:",
    disease_symptom_probability.shape
)

In [ ]:
disease_symptom_probability = (
    disease_symptom_probability
    .reindex(range(len(label_encoder.classes_)))
    .fillna(0)
)

print("Final shape:", disease_symptom_probability.shape)

**Symptom-Input Function**

In [ ]:
def create_symptom_input(
    confirmed_symptoms=None,
    denied_symptoms=None
):
    confirmed_symptoms = confirmed_symptoms or []
    denied_symptoms = denied_symptoms or []

    # -1 = UNKNOWN, 1 = YES, 0 = NO
    input_data = pd.DataFrame(
        -1.0,
        index=[0],
        columns=X.columns,
        dtype=np.float32
    )

    for symptom in confirmed_symptoms:
        if symptom in input_data.columns:
            input_data.loc[0, symptom] = 1.0

    for symptom in denied_symptoms:
        if symptom in input_data.columns:
            input_data.loc[0, symptom] = 0.0

    return input_data

**Shannon Entropy**

In [ ]:
def calculate_entropy(probabilities):

    probabilities = np.asarray(probabilities)

    # Remove zero probabilities
    probabilities = probabilities[probabilities > 0]

    entropy = -np.sum(
        probabilities * np.log2(probabilities)
    )

    return entropy

In [ ]:
example_probabilities = np.array([
    0.40,
    0.35,
    0.25
])

entropy = calculate_entropy(example_probabilities)

print("Entropy:", entropy)

**Calculate Next best question**

In [ ]:
def find_next_best_question(
    current_input,
    answered_symptoms,
    top_k=50
):

    # --------------------------------
    # Current disease probabilities
    # --------------------------------
    current_probabilities = (
        model_pipeline
        .predict_proba(current_input)[0]
    )

    # --------------------------------
    # Current uncertainty
    # --------------------------------
    current_entropy = calculate_entropy(
        current_probabilities
    )

    # --------------------------------
    # Select top important symptoms
    # --------------------------------
    important_symptoms = (
        feature_importance_df
        .head(top_k)["Symptom"]
        .tolist()
    )

    # --------------------------------
    # Remove already answered symptoms
    # --------------------------------
    unknown_symptoms = [
        symptom
        for symptom in important_symptoms
        if symptom not in answered_symptoms
    ]

    # If no symptoms are left
    if not unknown_symptoms:
        return pd.DataFrame()

    results = []

    # --------------------------------
    # Evaluate each unknown symptom
    # --------------------------------
    for symptom in unknown_symptoms:

        # ==============================
        # Simulate YES
        # ==============================
        yes_input = current_input.copy()
        yes_input.loc[0, symptom] = 1

        yes_probabilities = (
            model_pipeline
            .predict_proba(yes_input)[0]
        )

        yes_entropy = calculate_entropy(
            yes_probabilities
        )

        # ==============================
        # Simulate NO
        # ==============================
        no_input = current_input.copy()
        no_input.loc[0, symptom] = 0

        no_probabilities = (
            model_pipeline
            .predict_proba(no_input)[0]
        )

        no_entropy = calculate_entropy(
            no_probabilities
        )

        # ==============================
        # Estimate P(YES)
        # ==============================
        symptom_probabilities = (
            disease_symptom_probability[symptom]
            .values
        )

        p_yes = np.sum(
            current_probabilities *
            symptom_probabilities
        )

        p_yes = np.clip(
            p_yes,
            0,
            1
        )

        p_no = 1 - p_yes

        # ==============================
        # Expected Entropy
        # ==============================
        expected_entropy = (
            p_yes * yes_entropy
            +
            p_no * no_entropy
        )

        # ==============================
        # Information Gain
        # ==============================
        information_gain = (
            current_entropy
            -
            expected_entropy
        )

        results.append({
            "Symptom": symptom,
            "P(Yes)": p_yes,
            "P(No)": p_no,
            "Current Entropy": current_entropy,
            "Expected Entropy": expected_entropy,
            "Information Gain": information_gain
        })

    # --------------------------------
    # Sort by highest information gain
    # --------------------------------
    results_df = pd.DataFrame(results)

    results_df = results_df.sort_values(
        by="Information Gain",
        ascending=False
    ).reset_index(drop=True)

    return results_df

**Display the best question**

In [ ]:
def adaptive_disease_prediction(
    initial_symptoms,
    confidence_threshold=0.80,
    max_questions=7
):

    confirmed_symptoms = set(initial_symptoms)
    denied_symptoms = set()
    answered_symptoms = set(initial_symptoms)

    question_count = 0

    current_input = create_symptom_input(
        confirmed_symptoms=confirmed_symptoms,
        denied_symptoms=denied_symptoms
    )

    print("=" * 60)
    print("MEDIGUIDE AI")
    print("ADAPTIVE SYMPTOM ASSESSMENT")
    print("=" * 60)

    print("\nInitial Symptoms:")

    for symptom in confirmed_symptoms:
        print("✓", symptom)

    while question_count < max_questions:

        # --------------------------------
        # Current disease probabilities
        # --------------------------------
        probabilities = model_pipeline.predict_proba(
            current_input
        )[0]

        # Highest probability disease
        top_index = np.argmax(probabilities)

        top_disease = label_encoder.inverse_transform(
            [top_index]
        )[0]

        top_probability = probabilities[top_index]

        print("\n" + "-" * 60)

        print(
            f"Current Prediction: {top_disease}"
        )

        print(
            f"Confidence: {top_probability * 100:.2f}%"
        )

        # --------------------------------
        # Stopping condition
        # --------------------------------
        if top_probability >= confidence_threshold:

            print("\nConfidence threshold reached.")
            break

        # --------------------------------
        # Find next best question
        # --------------------------------
        question_results = find_next_best_question(
            current_input,
            answered_symptoms,
            top_k=50
        )

        if question_results.empty:

            print(
                "\nNo unanswered candidate symptoms remaining."
            )
            break

        best_question = question_results.iloc[0]

        symptom = best_question["Symptom"]

        print("\nNext Best Question:")

        print(
            f"Do you have '{symptom}'?"
        )

        print(
            "Information Gain:",
            round(
                best_question["Information Gain"],
                4
            )
        )

        # --------------------------------
        # Get user answer
        # --------------------------------
        answer = input(
            "Enter yes/no: "
        ).strip().lower()

        while answer not in ["yes", "no", "y", "n"]:

            print("Please enter yes or no.")

            answer = input(
                "Enter yes/no: "
            ).strip().lower()

        # --------------------------------
        # Update symptom state
        # --------------------------------
        if answer in ["yes", "y"]:

            confirmed_symptoms.add(symptom)

            current_input.loc[
                0, symptom
            ] = 1

            print(
                f"✓ Confirmed: {symptom}"
            )

        else:

            denied_symptoms.add(symptom)

            current_input.loc[
                0, symptom
            ] = 0

            print(
                f"✗ Denied: {symptom}"
            )

        answered_symptoms.add(symptom)

        question_count += 1

    # --------------------------------
    # FINAL PREDICTION
    # --------------------------------

    final_probabilities = (
        model_pipeline.predict_proba(
            current_input
        )[0]
    )

    final_index = np.argmax(
        final_probabilities
    )

    final_disease = label_encoder.inverse_transform(
        [final_index]
    )[0]

    final_confidence = (
        final_probabilities[final_index]
    )

    print("\n" + "=" * 60)
    print("FINAL RESULT")
    print("=" * 60)

    # --------------------------------
    # Confidence >= 80%
    # --------------------------------
    if final_confidence >= confidence_threshold:

        print(
            "Predicted Disease:",
            final_disease
        )

        print(
            f"Confidence: "
            f"{final_confidence * 100:.2f}%"
        )

    # --------------------------------
    # Confidence < 80%
    # --------------------------------
    else:

        print(
            f"Confidence is below "
            f"{confidence_threshold * 100:.0f}%."
        )

        print("\nTop 3 Possible Conditions:")

        top_3_indices = np.argsort(
            final_probabilities
        )[-3:][::-1]

        for rank, index in enumerate(
            top_3_indices,
            start=1
        ):

            disease = label_encoder.inverse_transform(
                [index]
            )[0]

            probability = (
                final_probabilities[index] * 100
            )

            print(
                f"{rank}. {disease} "
                f"→ {probability:.2f}%"
            )

    print(
        f"\nQuestions Asked: "
        f"{question_count}"
    )

    return {
        "disease": final_disease,
        "confidence": final_confidence,
        "questions_asked": question_count,
        "confirmed_symptoms": list(
            confirmed_symptoms
        ),
        "denied_symptoms": list(
            denied_symptoms
        )
    }

In [ ]:
# Show available symptoms
print("Please enter your starting symptoms.")
print("Use the symptom names exactly as they appear in the dataset.\n")

user_input = input(
    "Enter symptoms separated by commas: "
)

# Convert user input into a list
initial_symptoms = [
    symptom.strip().lower()
    for symptom in user_input.split(",")
    if symptom.strip()
]

# Validate symptoms
valid_symptoms = []
invalid_symptoms = []

for symptom in initial_symptoms:

    if symptom in X.columns:
        valid_symptoms.append(symptom)
    else:
        invalid_symptoms.append(symptom)

# Display invalid symptoms
if invalid_symptoms:
    print("\nThese symptoms were not found in the dataset:")

    for symptom in invalid_symptoms:
        print("-", symptom)

# Check whether at least one valid symptom was entered
if not valid_symptoms:
    print("\nNo valid symptoms entered.")

else:
    print("\nStarting symptoms:")

    for symptom in valid_symptoms:
        print("✓", symptom)

    result = adaptive_disease_prediction(
        initial_symptoms=valid_symptoms,
        confidence_threshold=0.80,
        max_questions=7
    )